In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np

PROJECT_ROOT = Path().resolve().parent  # notebooks/ -> project root
DB_PATH = PROJECT_ROOT / "data" / "cineanalytica.db"
conn = sqlite3.connect(DB_PATH)

In [2]:
df = pd.read_sql_query("SELECT * FROM movie_features", conn)
df.shape, df.head()

((4803, 33),
    movie_id                                     title     budget     revenue  \
 0     19995                                    Avatar  237000000  2787965087   
 1       285  Pirates of the Caribbean: At World's End  300000000   961000000   
 2    206647                                   Spectre  245000000   880674609   
 3     49026                     The Dark Knight Rises  250000000  1084939099   
 4     49529                               John Carter  260000000   284139100   
 
    runtime  popularity  vote_average  vote_count  release_year  release_month  \
 0    162.0  150.437577           7.2       11800        2009.0           12.0   
 1    169.0  139.082615           6.9        4500        2007.0            5.0   
 2    148.0  107.376788           6.3        4466        2015.0           10.0   
 3    165.0  112.312950           7.6        9106        2012.0            7.0   
 4    132.0   43.926995           6.1        2124        2012.0            3.0   
 
    .

In [3]:
if "log_revenue" not in df.columns:
    df["log_revenue"] = np.log1p(df["revenue"])

# quick sanity
df[["revenue","log_revenue"]].describe()

,revenue,log_revenue
count,4.803000e+03,4803.000000
mean,8.226064e+07,12.220768
std,1.628571e+08,8.157087
min,0.000000e+00,0.000000
25%,0.000000e+00,0.000000
50%,1.917000e+07,16.768857
75%,9.291719e+07,18.347219
max,2.787965e+09,21.748578


In [4]:
df.isna().sum().sort_values(ascending=False).head(20)

runtime            2
release_month      1
release_year       1
Science Fiction    0
Foreign            0
History            0
Horror             0
Music              0
Mystery            0
Romance            0
movie_id           0
Family             0
TV Movie           0
Thriller           0
War                0
Western            0
log_star_power     0
Fantasy            0
Documentary        0
Drama              0
dtype: int64

In [5]:
# Minimal imputation for small missing counts
for col in ["runtime", "release_year", "release_month"]:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Sanity check
df[["runtime","release_year","release_month"]].isna().sum()

runtime          0
release_year     0
release_month    0
dtype: int64

In [6]:
target = "log_revenue"

# Columns we should NOT use as features
drop_cols = ["movie_id", "title", "revenue", target]

# Keep only numeric columns for ML (genres are numeric 0/1)
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
X = X.select_dtypes(include=[np.number])

y = df[target].astype(float)

X.shape, y.shape

((4803, 30), (4803,))

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

rmse, mae, r2

(8.272281713725352, 7.47769540804945, -0.0007763662747009015)

In [ ]:
##Linear Regression

In [9]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

pred_lin = lin_reg.predict(X_test)

rmse_lin = np.sqrt(mean_squared_error(y_test, pred_lin))
mae_lin = mean_absolute_error(y_test, pred_lin)
r2_lin = r2_score(y_test, pred_lin)

rmse_lin, mae_lin, r2_lin

(6.695499232567198, 5.46075529083111, 0.3443797817289126)

In [10]:
## Random Forest

In [11]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
mae_rf = mean_absolute_error(y_test, pred_rf)
r2_rf = r2_score(y_test, pred_rf)

rmse_rf, mae_rf, r2_rf

(5.13969971933919, 3.1371766843845243, 0.6136669150052715)

In [12]:
## XGBoost

In [13]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

pred_xgb = xgb.predict(X_test)

rmse_xgb = np.sqrt(mean_squared_error(y_test, pred_xgb))
mae_xgb = mean_absolute_error(y_test, pred_xgb)
r2_xgb = r2_score(y_test, pred_xgb)

rmse_xgb, mae_xgb, r2_xgb

(5.1005910972770465, 3.131453921348165, 0.6195238609221234)

In [14]:
df["log_budget"] = np.log1p(df["budget"])

In [15]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

pred_xgb = xgb.predict(X_test)

rmse_xgb = np.sqrt(mean_squared_error(y_test, pred_xgb))
mae_xgb = mean_absolute_error(y_test, pred_xgb)
r2_xgb = r2_score(y_test, pred_xgb)

rmse_xgb, mae_xgb, r2_xgb

(5.1005910972770465, 3.131453921348165, 0.6195238609221234)